In [1]:
import os
import sys
import pandas as pd
import numpy as np

In [2]:
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
from lightning.pytorch.loggers import TensorBoardLogger

In [3]:
from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import MAE, SMAPE, PoissonLoss, QuantileLoss
from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters

In [4]:
# Load the data
dir_path = "./"

# Append the parent directory to sys.path
sys.path.append(os.path.join(dir_path, "./"))

# File path to the data file
raw_file_path = os.path.join(dir_path, 'data', "raw_data.parquet")  
print(raw_file_path)

# Read the data 
raw_df = pd.read_parquet(raw_file_path)

# Convert all Prices which are zero to NaN for interpolation 
raw_df['Price'] = raw_df['Price'].replace(0, np.nan)

./data/raw_data.parquet


In [5]:
# Function to impute missing prices based on the hierarchical strategy, starting with linear interpolation
def hierarchical_imputation(df):
    # Step 1: Apply linear interpolation for each combination of Product, Client, and Warehouse over time (Week)
    df['Imputed_Price'] = df.groupby(['Product', 'Client', 'Warehouse'])['Price'].apply(lambda group: group.interpolate(method='linear')).values

    # Step 2: If missing, impute by same Product, Client, and Warehouse (mean of the group)
    # df['Imputed_Price'] = df.groupby(['Product', 'Client', 'Warehouse'])['Imputed_Price'].transform(lambda x: x.fillna(x.mean()))

    # Step 3: If missing, impute by same Product and Client across all Warehouses
    # df['Imputed_Price'] = df.groupby(['Product', 'Client'])['Imputed_Price'].transform(lambda x: x.fillna(x.mean()))

    # Step 4: If missing, impute by same Product in the same Warehouse but across all Clients
    # df['Imputed_Price'] = df.groupby(['Product', 'Warehouse'])['Imputed_Price'].transform(lambda x: x.fillna(x.mean()))

    # Step 5: If still missing, impute by same Product across all Clients and Warehouses
    # df['Imputed_Price'] = df.groupby(['Product'])['Imputed_Price'].transform(lambda x: x.fillna(x.mean()))

    return df

In [ ]:
# Apply hierarchical imputation
raw_df['Imputed_Price'] = raw_df.groupby(['Product', 'Client', 'Warehouse'])['Price'].apply(
    lambda group: group.interpolate(method='linear')
)

# Check for any remaining missing prices after imputation
missing_after_imputation = raw_df['Imputed_Price'].isnull().sum()
print(f"Remaining missing prices after imputation: {missing_after_imputation}")


In [ ]:
result_df.head()

In [ ]:
unique_id = '0_1_367'
check_df = result_df[(result_df['client_warehouse_product_id'] == unique_id)]
check_df.tail(15)

In [6]:
np.random.seed(42)

def generate_1min_data(start_date='2021-06-01', 
                       end_date='2021-06-02'):
    """Generate a synthetic random-walk price series and volume data for the 1-min timeframe."""
    # Create a range of business days, then for each day generate 1-min timestamps during market hours
    trading_days = pd.date_range(start_date, end_date, freq='B')  # freq='B' → Business days
    timestamps = []
    
    for day in trading_days:
        # Create a date range for each day, from 09:30 to 16:00, in 1-min increments
        day_str = day.strftime('%Y-%m-%d')
        day_times = pd.date_range(day_str + ' 09:30', 
                                  day_str + ' 16:00', 
                                  freq='1min')
        timestamps.append(day_times)
    
    # Concatenate all timestamps into a single index
    all_times = pd.DatetimeIndex(np.concatenate(timestamps))
    
    # Generate a random walk for "Close" prices
    n = len(all_times)
    price_changes = np.random.normal(loc=0.0, scale=0.1, size=n).cumsum()
    base_price = 100.0  # Starting price
    close_prices = base_price + price_changes
    
    # Synthetic "Open", "High", and "Low" around the close price
    open_prices = close_prices - np.random.uniform(0.0, 0.05, size=n)
    high_prices = np.maximum(open_prices, close_prices) + np.random.uniform(0.0, 0.05, size=n)
    low_prices  = np.minimum(open_prices, close_prices) - np.random.uniform(0.0, 0.05, size=n)
    
    # Synthetic volume
    volumes = np.random.randint(100, 500, size=n)
    
    df_1min = pd.DataFrame({
        'Open': open_prices,
        'High': high_prices,
        'Low': low_prices,
        'Close': close_prices,
        'Volume': volumes
    }, index=all_times)
    
    return df_1min

df_1min = generate_1min_data()
print("1-Minute Synthetic Data (head):")
print(df_1min.head())

1-Minute Synthetic Data (head):
                           Open        High         Low       Close  Volume
2021-06-01 09:30:00  100.006024  100.082639   99.994405  100.049671     158
2021-06-01 09:31:00   99.999234  100.080601   99.985184  100.035845     412
2021-06-01 09:32:00  100.060286  100.132447  100.020112  100.100614     139
2021-06-01 09:33:00  100.219978  100.283614  100.173516  100.252917     336
2021-06-01 09:34:00  100.194888  100.232834  100.174633  100.229501     485
